# Tests del modelo (M3) - `tests/test_model.py`

**Archivo a testear:** `src/qap/model.py`
**Tests:** `tests/test_model.py`

Los tests verifican que la formulación QUBO/Ising es correcta según lo que pide el plan:

- Para N=3 y N=4 se enumeran todas las asignaciones binarias $(2^{N^2}=512/65536)$ y se comprueba que el mínimo con $λ$ suficiente es una permutación óptima (24/32).
- Para cada permutación factible, la energía QUBO penalizada es igual al costo real `cost(perm)` (el término de penalización es cero).
- La relación $x = (1-Z)/2$ es correcta: $E_{QUBO}(x) = \langle x|H|x\rangle + offset$.
- La trampa del orden de bits: las cadenas de bits medidos en Qiskit are **little-endian** (el carácter más a la derecha = qubit 0), por lo que hay que invertirlos antes de mapear a $x_{i,j}$.
- El umbral de penalización $λ^* > 0$.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))

import numpy as np
import itertools
from qiskit.quantum_info import Statevector

from qap.model import build_qp, to_qubo, to_ising, decode, decode_x, x_from_bits, penalty_threshold, is_feasible, true_cost, penalized_energy
from qap.types import Instance, cost

## Setup con las instancias de test (primeros 4x4 de qap10)

In [ ]:
F = np.array([[0,10,4,8],[0,0,6,0],[0,0,0,0],[0,0,0,0]])
D = np.array([[0,1,2,3],[1,0,1,2],[2,1,0,1],[3,2,1,0]])

def instance(n):
    return Instance(f'test{n}', n, F[:n,:n].copy(), D[:n,:n].copy())

## test_qubo_energy_equals_cost_on_permutations (N=3,4)

In [ ]:
for n in [3,4]:
    inst = instance(n)
    qubo = to_qubo(build_qp(inst), lam=penalty_threshold(inst)*2)
    ok = True
    for perm in itertools.permutations(range(n)):
        x = np.zeros((n,n), dtype=int)
        x[np.arange(n), list(perm)] = 1
        e = penalized_energy(x.ravel(), qubo)
        c = true_cost(perm, inst)
        if not np.isclose(e, c):
            ok = False
            print('FAIL', n, perm, e, c)
            break
    print(f'N={n}: QUBO energy == cost on all permutations: {ok}')

## test_qp_reaches_control_optimum (N=3,4) - fuerza bruta sobre todas las asignaciones binarias

In [ ]:
for n, optimum in [(3,24),(4,32)]:
    inst = instance(n)
    qubo = to_qubo(build_qp(inst), lam=penalty_threshold(inst)*2)
    vectors = ((np.arange(2**(n*n))[:,None]) >> np.arange(n*n)) & 1
    energies = np.array([qubo.objective.evaluate(v.astype(int)) for v in vectors])
    idx = int(np.argmin(energies))
    perm = decode_x(vectors[idx], n)
    print(f'N={n}: argmin perm={perm}, cost={cost(perm, inst) if perm else None}, optimum={optimum}, feasible={is_feasible(perm,n)}')

## test_ising_offset_matches_ground_energy (N=3,4) - $x = (1-Z)/2$

In [ ]:
for n in [3,4]:
    inst = instance(n)
    qubo = to_qubo(build_qp(inst), lam=penalty_threshold(inst)*2)
    op, offset = to_ising(qubo)
    ok = True
    for perm in list(itertools.permutations(range(n))[:10]):
        x = np.zeros(n*n)
        for i,p in enumerate(perm):
            x[i*n + p] = 1
        qv = qubo.objective.evaluate(x)
        sv = Statevector.from_label(''.join(str(int(b)) for b in x[::-1]))  # little-endian
        hv = float(np.real_if_close(sv.expectation_value(op)))
        if not np.isclose(hv + offset, qv):
            ok = False
            print('FAIL', n, perm)
            break
    print(f'N={n}: <x|H|x>+offset == QUBO: {ok}')

## test_decode_little_endian - orden de bits

In [ ]:
x = np.eye(3).ravel().astype(int)
bits = ''.join(str(int(b)) for b in x[::-1])
print('bits:', bits)
print('decode:', decode(bits, 3))
print('x_from_bits:', x_from_bits(bits))
print('inválido (muy corto):', decode('0', 3))

## test_penalty_threshold_positive

In [ ]:
for n in [3,4]:
    print(f'N={n} lambda*:', penalty_threshold(instance(n)))

## Ejecutar todos los tests con pytest

In [ ]:
!cd .. && $env:PYTHONPATH='src' && uvx --with numpy --with pytest --with qiskit --with qiskit-aer --with qiskit-algorithms --with qiskit-optimization --with scipy pytest tests/test_model.py -v 2>&1 | tail -15

OJO: 
nosotros podemos utilizar el exact_optimum.json para  comprobar que la penalización automática de Qiskit coinciden con los del JSON. Así, si cualquiera de los dos cambia algo, se detecta. Pero por ahora no lo vamos a poner.
